# CMEGNets — Implementation / 구현

**Paper**: Guesmi, B., Daghrir, J., Moloney, D., Espinosa-Aranda, J. L., & Hervas-Martin, E. (2026). *CMEGNets: A self-supervised framework for coronal mass ejection detection & region segmentation*. Advances in Space Research, 77, 7455–7483.

이 노트북은 CMEGNets의 **세 가지 핵심 구성요소**를 NumPy·PyTorch로 구현하고 합성 데이터로 검증한다. / This notebook implements the **three core building blocks** of CMEGNets from scratch (NumPy/PyTorch) and validates them on synthetic data.

1. **NT-Xent contrastive loss** — SimCLR의 학습 손실을 NumPy로 재현 / Reproduce SimCLR's loss in NumPy.
2. **Reference-based cosine classification** — 평균 CME/non-CME 임베딩과의 거리로 분류 / Classify by distance to mean reference embeddings.
3. **Mahalanobis-based anomaly segmentation** — quiet-Sun Gaussian으로부터 이상치 scoring → 픽셀 단위 CME 마스크 / Anomaly scoring against a quiet-Sun Gaussian baseline → pixel-level CME masks.

실제 LASCO 데이터를 ~230만 장 규모로 pretraining하는 SimCLR 자체는 GPU cluster 규모가 필요하므로, 여기서는 **개념 검증(concept validation)**에 집중한다. / Full-scale SimCLR pre-training on 2.3M LASCO frames needs GPU clusters; this notebook focuses on **concept validation**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.spatial.distance import mahalanobis as scipy_mahalanobis
from scipy.ndimage import gaussian_filter
from sklearn.covariance import EmpiricalCovariance
from sklearn.metrics import f1_score, accuracy_score

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch unavailable — mini U-Net section will be skipped.')

np.random.seed(42)
if TORCH_AVAILABLE:
    torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

---
## Part 1: NT-Xent Contrastive Loss / NT-Xent 대조 손실

SimCLR (Chen et al. 2020)의 학습 손실은 다음과 같다. / SimCLR's learning objective:

$$\ell(i,j) = -\log \frac{\exp\!\bigl(\operatorname{sim}(z_i, z_j)/\tau\bigr)}{\sum_{k=1}^{2N}\mathbf{1}_{[k\neq i]}\exp\!\bigl(\operatorname{sim}(z_i, z_k)/\tau\bigr)}$$

여기서 $\operatorname{sim}(u,v) = u \cdot v / (\|u\|\|v\|)$는 코사인 유사도, $\tau$는 temperature, $2N$은 배치 내 augmented 샘플 총 수이다. / $\operatorname{sim}$ is cosine similarity, $\tau$ is temperature, $2N$ = augmented batch size.

**직관 / Intuition**: 같은 이미지의 다른 view $(i,j)$를 가깝게, 다른 이미지들을 멀리 민다. / Pull positive pairs $(i,j)$ together, push all other samples apart.

논문에서는 $\tau = 0.5$로 설정. / Paper uses $\tau = 0.5$.

In [ ]:
def nt_xent_loss(z: np.ndarray, tau: float = 0.5) -> float:
    """Compute the NT-Xent contrastive loss over an augmented batch.

    Embeddings are assumed to be stacked as 2N rows where rows 2k and 2k+1
    form a positive pair (the two augmentations of the k-th source image).

    Args:
        z: Array of shape (2N, d). Embeddings for the 2N augmented samples.
        tau: Temperature parameter (paper uses 0.5).

    Returns:
        Mean NT-Xent loss across the batch.
    """
    two_n, _ = z.shape
    z_norm = z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-12)
    sim = z_norm @ z_norm.T  # (2N, 2N) cosine similarity matrix
    sim_scaled = sim / tau
    np.fill_diagonal(sim_scaled, -np.inf)  # exclude self-similarity

    # Positive-pair indices: sample 2k ↔ sample 2k+1
    pos_idx = np.arange(two_n)
    pos_idx = pos_idx ^ 1  # XOR with 1 flips last bit -> pairs (0,1),(2,3),...

    logsumexp = np.log(np.exp(sim_scaled).sum(axis=1) + 1e-12)
    pos_sim = sim_scaled[np.arange(two_n), pos_idx]
    losses = -(pos_sim - logsumexp)
    return float(losses.mean())


# Smoke test: random embeddings should give loss ≈ log(2N-1) near τ = 0.5
N = 16
d = 128
z_random = np.random.randn(2 * N, d)
z_aligned = np.repeat(np.random.randn(N, d), 2, axis=0) + 0.01 * np.random.randn(2 * N, d)

loss_rand = nt_xent_loss(z_random, tau=0.5)
loss_aligned = nt_xent_loss(z_aligned, tau=0.5)
print(f'Loss with random embeddings    : {loss_rand:.3f}')
print(f'Loss with aligned positive pairs: {loss_aligned:.3f}')
print(f'  → Aligned loss should be ≪ random loss (positive pairs are easy to identify).')

### 1.1 Temperature sweep / 온도 파라미터 스윕

Temperature $\tau$는 negative sample에 부여할 "가혹도(hardness)"를 조절한다. / Temperature controls how hard the loss pushes negatives.

- 작은 $\tau$: sharper softmax → 가장 비슷한 negative를 집중적으로 밀어냄 / Concentrates on hardest negatives.
- 큰 $\tau$: flatter softmax → 모든 negative를 균등하게 처리 / Uniform push on all negatives.

In [ ]:
taus = np.logspace(-1.5, 0.5, 9)
losses_rand = [nt_xent_loss(z_random, tau=t) for t in taus]
losses_aligned = [nt_xent_loss(z_aligned, tau=t) for t in taus]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(taus, losses_rand, 'o-', label='random embeddings')
ax.semilogx(taus, losses_aligned, 's-', label='aligned positive pairs')
ax.axvline(0.5, color='red', linestyle='--', label=r'paper $\tau = 0.5$')
ax.set_xlabel(r'Temperature $\tau$')
ax.set_ylabel('NT-Xent loss')
ax.set_title('NT-Xent loss vs. temperature (N=16, d=128)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 2: Reference-Based Cosine Classification / 참조 기반 코사인 분류

논문 식 (6)의 분류 규칙을 구현한다. / Implements the classification rule in Eq. (6).

$$d_{\cos}(z_t, z_r) = 1 - \frac{z_t \cdot z_r}{\|z_t\|_2 \|z_r\|_2}$$

$$\hat{y} = \begin{cases} \text{CME} & \text{if } d_{\cos}(z_t, z_c) \le d_{\cos}(z_t, z_{nc}) \\ \text{non-CME} & \text{otherwise}\end{cases}$$

여기서 $z_c, z_{nc}$는 CME·non-CME 레퍼런스 임베딩의 평균이다. / $z_c, z_{nc}$ are the mean CME/non-CME reference embeddings.

In [ ]:
def cosine_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine distance broadcast over batched vectors.

    Args:
        a: Array of shape (..., d).
        b: Array of shape (d,) or broadcastable with a.

    Returns:
        Cosine distance with shape a.shape[:-1].
    """
    a_norm = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)
    b_norm = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-12)
    return 1.0 - np.sum(a_norm * b_norm, axis=-1)


def classify_by_reference(
    z_test: np.ndarray, z_ref_cme: np.ndarray, z_ref_nc: np.ndarray
) -> np.ndarray:
    """Binary classify test embeddings by nearer reference cluster.

    Args:
        z_test: Test embeddings, shape (M, d).
        z_ref_cme: CME reference embeddings, shape (R_c, d).
        z_ref_nc: Non-CME reference embeddings, shape (R_nc, d).

    Returns:
        Integer labels of shape (M,): 1 for CME, 0 for non-CME.
    """
    mu_cme = z_ref_cme.mean(axis=0)
    mu_nc = z_ref_nc.mean(axis=0)
    d_cme = cosine_distance(z_test, mu_cme)
    d_nc = cosine_distance(z_test, mu_nc)
    return (d_cme <= d_nc).astype(int)


# Synthesize two well-separated clusters in 128-d (mimicking SimCLR output)
D = 128


def gen_cluster(center: np.ndarray, n: int, scale: float = 0.3) -> np.ndarray:
    """Generate n points around `center` with Gaussian noise."""
    return center + scale * np.random.randn(n, len(center))


mu_cme_true = np.random.randn(D)
mu_nc_true = np.random.randn(D)

ref_cme = gen_cluster(mu_cme_true, 60)
ref_nc = gen_cluster(mu_nc_true, 20)

test_cme = gen_cluster(mu_cme_true, 200)
test_nc = gen_cluster(mu_nc_true, 200)
z_test = np.vstack([test_cme, test_nc])
y_test = np.array([1] * 200 + [0] * 200)

y_hat = classify_by_reference(z_test, ref_cme, ref_nc)
print(f'Reference-based accuracy: {accuracy_score(y_test, y_hat):.4f}')
print(f'Reference-based F1      : {f1_score(y_test, y_hat):.4f}')

---
## Part 3: Mahalanobis Anomaly Detection / Mahalanobis 이상치 탐지

각 패치 위치 $i$에서 "quiet Sun" feature의 다변량 Gaussian을 적합하고, 테스트 프레임의 deviation을 Mahalanobis 거리로 측정한다. / Fit a multivariate Gaussian per patch on quiet-Sun features; measure deviation of a test frame by Mahalanobis distance.

$$\mu_i = \frac{1}{N}\sum_{j=1}^N f_i^j, \qquad \Sigma_i = \frac{1}{N-1}\sum_{j=1}^N (f_i^j - \mu_i)(f_i^j - \mu_i)^\top$$
$$d_M(f, \mu_i) = \sqrt{(f - \mu_i)^\top \Sigma_i^{-1} (f - \mu_i)}$$

In [ ]:
def fit_gaussian(features: np.ndarray, ridge: float = 1e-3) -> tuple:
    """Fit mean and covariance (with ridge) to a set of feature vectors.

    Args:
        features: Array of shape (N, d) — samples for one patch location.
        ridge: Diagonal regularizer ensuring `Sigma` is invertible.

    Returns:
        A tuple (mu, Sigma_inv) where mu has shape (d,) and Sigma_inv is (d, d).
    """
    mu = features.mean(axis=0)
    centered = features - mu
    sigma = centered.T @ centered / max(features.shape[0] - 1, 1)
    sigma += ridge * np.eye(sigma.shape[0])
    return mu, np.linalg.inv(sigma)


def mahalanobis_distance(f: np.ndarray, mu: np.ndarray, sigma_inv: np.ndarray) -> np.ndarray:
    """Vectorised Mahalanobis distance for a batch of query vectors.

    Args:
        f: Array of shape (..., d) — query vectors.
        mu: Array of shape (d,) — distribution mean.
        sigma_inv: Array of shape (d, d) — inverse covariance.

    Returns:
        Array of shape (...,) with Mahalanobis distances.
    """
    diff = f - mu
    # Einsum over the last axis to preserve leading batch dims
    quad = np.einsum('...i,ij,...j->...', diff, sigma_inv, diff)
    return np.sqrt(np.clip(quad, 0.0, None))


# Verification against scipy.spatial.distance.mahalanobis
d = 3
normal = np.random.randn(500, d) * np.array([1.0, 2.0, 0.5])
mu_hat, sigma_inv_hat = fit_gaussian(normal)

query = np.array([3.0, -2.0, 1.0])
ours = mahalanobis_distance(query, mu_hat, sigma_inv_hat)
scipy_val = scipy_mahalanobis(query, mu_hat, sigma_inv_hat)
print(f'Our implementation: {ours:.4f}')
print(f'scipy reference  : {scipy_val:.4f}')
assert np.isclose(ours, scipy_val, atol=1e-6), 'Mahalanobis mismatch vs scipy'
print('✓ Matches scipy.spatial.distance.mahalanobis.')

---
## Part 4: Synthetic LASCO-like Data / 합성 LASCO 유사 데이터

실제 LASCO C2 영상 없이도 CMEGNets 파이프라인을 시연하기 위해, 다음 요소를 합성한다. / To demonstrate the CMEGNets pipeline without real LASCO data, we synthesize:

- **Occulter disc** (중심 원반) — 코로나그래프의 태양 차단판 / The coronagraph occulter.
- **K-corona radial falloff** — 중심으로부터 멀어질수록 어두워짐 / Brightness falling radially.
- **Streamers** — 정상적 radial feature / Quasi-stationary radial features.
- **Noise** — Gaussian + salt-and-pepper / Stochastic detector noise.
- **CME anomaly** — 팬 모양의 밝은 lobe (halo/partial-halo 모사) / Fan-shaped bright lobe.

*(This is an illustrative toy; it is not meant to reproduce photometric fidelity of real LASCO.)*

In [ ]:
IMG_SIZE = 128


def make_polar_coords(size: int) -> tuple:
    """Return meshgrid of (radius_normalized, angle_rad) of shape (size, size)."""
    y, x = np.mgrid[:size, :size]
    cx, cy = size / 2, size / 2
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2) / (size / 2)
    theta = np.arctan2(y - cy, x - cx)
    return r, theta


R, THETA = make_polar_coords(IMG_SIZE)


def quiet_corona(noise_scale: float = 0.05, seed: int | None = None) -> np.ndarray:
    """Generate a synthetic quiet-Sun LASCO frame.

    Args:
        noise_scale: Std-dev of additive Gaussian noise.
        seed: Optional random seed for reproducibility.

    Returns:
        Normalised [0,1] image of shape (IMG_SIZE, IMG_SIZE).
    """
    rng = np.random.default_rng(seed)
    img = np.exp(-2.0 * R)  # radial K-corona falloff
    # Streamers: a few high-intensity ridges at random angles
    for _ in range(3):
        theta0 = rng.uniform(-np.pi, np.pi)
        width = rng.uniform(0.1, 0.25)
        amp = rng.uniform(0.08, 0.18)
        img += amp * np.exp(-((THETA - theta0) / width) ** 2) * np.exp(-1.5 * R)
    img += noise_scale * rng.standard_normal(img.shape)
    occulter = R < 0.18
    img[occulter] = 0.0
    return np.clip(img, 0, 1)


def cme_frame(
    cme_theta0: float = 0.3, cme_width: float = 0.5, cme_amp: float = 0.6, seed: int | None = None
) -> tuple:
    """Generate a LASCO frame with a synthetic CME lobe.

    Args:
        cme_theta0: Central angle of the CME (radians).
        cme_width: Angular half-width of the CME (radians).
        cme_amp: Peak intensity of the CME addition.
        seed: Optional random seed.

    Returns:
        A tuple (image, ground_truth_mask).
    """
    rng = np.random.default_rng(seed)
    img = quiet_corona(seed=seed)
    cme = cme_amp * np.exp(-((THETA - cme_theta0) / cme_width) ** 2)
    cme *= np.where((R > 0.25) & (R < 0.9), 1.0, 0.0)
    cme *= np.exp(-((R - 0.55) / 0.25) ** 2)
    cme += 0.05 * rng.standard_normal(cme.shape)
    cme = np.clip(cme, 0, None)
    img_out = np.clip(img + cme, 0, 1.2)
    gt_mask = cme > 0.1
    return img_out, gt_mask


# Visualise a few frames
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(quiet_corona(seed=0), cmap='inferno')
axes[0].set_title('Quiet Sun #1')
axes[1].imshow(quiet_corona(seed=1), cmap='inferno')
axes[1].set_title('Quiet Sun #2')
img_cme, gt = cme_frame(cme_theta0=-0.5, cme_amp=0.7, seed=10)
axes[2].imshow(img_cme, cmap='inferno')
axes[2].set_title('CME frame')
axes[3].imshow(gt, cmap='gray')
axes[3].set_title('Ground-truth mask')
for a in axes:
    a.axis('off')
plt.tight_layout()
plt.show()

---
## Part 5: Pipeline on Synthetic Data / 합성 데이터에 파이프라인 적용

논문은 ResNet-18의 layer 1–3 feature(d=3)를 패치별로 사용한다. 여기서는 **간단한 3-채널 handcrafted feature**(픽셀 강도, 지역 평균, 지역 표준편차)를 사용해 동일한 논리를 시연한다. / The paper uses ResNet-18 layer-1–3 features (d=3) per patch. Here we substitute a **handcrafted 3-channel feature** (intensity, local mean, local std) to demonstrate the logic.

1. Quiet-Sun 데이터 200장으로 per-pixel Gaussian fit
2. 테스트 CME 프레임에서 Mahalanobis 거리 계산
3. Threshold → binary mask → Dice 측정

In [ ]:
def extract_patch_features(img: np.ndarray, win: int = 5) -> np.ndarray:
    """Extract a 3-channel feature per pixel: intensity, local mean, local std.

    Args:
        img: 2-D input image of shape (H, W).
        win: Kernel size used for local statistics.

    Returns:
        Feature tensor of shape (H, W, 3).
    """
    intensity = img
    mean = gaussian_filter(img, sigma=win / 2)
    mean_sq = gaussian_filter(img ** 2, sigma=win / 2)
    std = np.sqrt(np.clip(mean_sq - mean ** 2, 0, None))
    return np.stack([intensity, mean, std], axis=-1)


def fit_per_pixel_gaussian(stack: np.ndarray, ridge: float = 1e-3) -> tuple:
    """Fit an independent Gaussian per pixel location.

    Args:
        stack: Feature stack of shape (N, H, W, d) across N quiet-Sun frames.
        ridge: Diagonal regularisation added to each covariance.

    Returns:
        A tuple (mu, sigma_inv) with shapes (H, W, d) and (H, W, d, d).
    """
    n, h, w, d = stack.shape
    mu = stack.mean(axis=0)  # (H, W, d)
    centered = stack - mu
    # Covariance per pixel: (H, W, d, d)
    sigma = np.einsum('nhwi,nhwj->hwij', centered, centered) / max(n - 1, 1)
    sigma += ridge * np.eye(d)
    sigma_inv = np.linalg.inv(sigma)
    return mu, sigma_inv


def pixel_mahalanobis_map(features: np.ndarray, mu: np.ndarray, sigma_inv: np.ndarray) -> np.ndarray:
    """Compute a per-pixel Mahalanobis heat map."""
    diff = features - mu  # (H, W, d)
    quad = np.einsum('hwi,hwij,hwj->hw', diff, sigma_inv, diff)
    return np.sqrt(np.clip(quad, 0.0, None))


def dice_coefficient(pred: np.ndarray, gt: np.ndarray) -> float:
    """Binary Dice coefficient between two masks."""
    pred_b = pred.astype(bool)
    gt_b = gt.astype(bool)
    inter = np.logical_and(pred_b, gt_b).sum()
    total = pred_b.sum() + gt_b.sum()
    return 2.0 * inter / total if total > 0 else 0.0


# 1. Build quiet-Sun baseline
N_quiet = 200
quiet_stack = np.stack([extract_patch_features(quiet_corona(seed=s)) for s in range(N_quiet)])
print(f'Quiet-Sun feature stack shape: {quiet_stack.shape}')
mu_map, sinv_map = fit_per_pixel_gaussian(quiet_stack)

# 2. Score a test CME frame
test_img, gt_mask = cme_frame(cme_theta0=-0.3, cme_width=0.55, cme_amp=0.6, seed=101)
test_feats = extract_patch_features(test_img)
heat = pixel_mahalanobis_map(test_feats, mu_map, sinv_map)

# 3. Threshold and evaluate
threshold = np.percentile(heat, 95)
pred_mask = heat > threshold
# Suppress occulter region (known prior)
pred_mask[R < 0.2] = False
pred_mask[R > 0.95] = False

dice = dice_coefficient(pred_mask, gt_mask)
print(f'Threshold (95th pct) : {threshold:.3f}')
print(f'Dice vs GT mask      : {dice:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(test_img, cmap='inferno')
axes[0].set_title('Synthetic LASCO frame')
axes[1].imshow(heat, cmap='viridis')
axes[1].set_title('Mahalanobis heat map')
axes[2].imshow(pred_mask, cmap='gray')
axes[2].set_title(f'Predicted mask (Dice={dice:.2f})')
axes[3].imshow(gt_mask, cmap='gray')
axes[3].set_title('Ground-truth mask')
for a in axes:
    a.axis('off')
plt.tight_layout()
plt.show()

### 5.1 Threshold sweep / 임계값 스윕

Mahalanobis threshold를 변화시키며 Dice를 추적한다. / Sweep the Mahalanobis threshold and watch Dice respond.

In [ ]:
percentiles = np.linspace(80, 99.5, 25)
dices = []
for p in percentiles:
    thr = np.percentile(heat, p)
    m = heat > thr
    m[R < 0.2] = False
    m[R > 0.95] = False
    dices.append(dice_coefficient(m, gt_mask))

best_idx = int(np.argmax(dices))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(percentiles, dices, 'o-')
ax.axvline(percentiles[best_idx], color='red', linestyle='--',
           label=f'best @ pct={percentiles[best_idx]:.1f}, Dice={dices[best_idx]:.3f}')
ax.set_xlabel('Heat-map percentile threshold')
ax.set_ylabel('Dice')
ax.set_title('Threshold sweep on synthetic CME frame')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 6: Mini U-Net Architecture / 소형 U-Net 구조 (PyTorch)

논문에서는 의사 라벨로 U-Net을 80 epoch 훈련한다. 여기서는 **모델 구조만 선언**하고 forward-pass 출력 shape를 확인한다 (full training은 GPU 필요). / The paper fine-tunes a U-Net for 80 epochs on pseudo-labels. Here we only **declare the architecture** and verify forward shapes (full training needs a GPU).

구조 / Architecture: symmetric encoder–decoder + skip connections (Ronneberger et al. 2015).

In [ ]:
if TORCH_AVAILABLE:
    class DoubleConv(nn.Module):
        """Two 3x3 convolutions with BatchNorm + ReLU."""

        def __init__(self, in_ch: int, out_ch: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)

    class MiniUNet(nn.Module):
        """Small U-Net with 4 encoder/decoder levels for 256x256 inputs."""

        def __init__(self, in_ch: int = 1, out_ch: int = 1, base: int = 16):
            super().__init__()
            self.e1 = DoubleConv(in_ch, base)
            self.e2 = DoubleConv(base, base * 2)
            self.e3 = DoubleConv(base * 2, base * 4)
            self.e4 = DoubleConv(base * 4, base * 8)
            self.bot = DoubleConv(base * 8, base * 16)
            self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
            self.d4 = DoubleConv(base * 16, base * 8)
            self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
            self.d3 = DoubleConv(base * 8, base * 4)
            self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            self.d2 = DoubleConv(base * 4, base * 2)
            self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            self.d1 = DoubleConv(base * 2, base)
            self.head = nn.Conv2d(base, out_ch, 1)
            self.pool = nn.MaxPool2d(2)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            e1 = self.e1(x)
            e2 = self.e2(self.pool(e1))
            e3 = self.e3(self.pool(e2))
            e4 = self.e4(self.pool(e3))
            b = self.bot(self.pool(e4))
            d4 = self.d4(torch.cat([self.up4(b), e4], dim=1))
            d3 = self.d3(torch.cat([self.up3(d4), e3], dim=1))
            d2 = self.d2(torch.cat([self.up2(d3), e2], dim=1))
            d1 = self.d1(torch.cat([self.up1(d2), e1], dim=1))
            return self.head(d1)

    net = MiniUNet()
    dummy = torch.zeros(1, 1, 256, 256)
    out = net(dummy)
    n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
    print(f'Input  shape : {tuple(dummy.shape)}')
    print(f'Output shape : {tuple(out.shape)}')
    print(f'Params       : {n_params:,} ({n_params / 1e6:.2f}M)')
    print('Paper reports <12M parameters for the full pipeline.')
else:
    print('PyTorch unavailable; skipping mini U-Net section.')

---
## Part 7: Comparison with Library Equivalents / 라이브러리 구현 비교

우리 구현을 `sklearn.covariance.EmpiricalCovariance`, `scipy.spatial.distance.mahalanobis`와 교차 검증한다. / Cross-check our implementation against sklearn/scipy library equivalents.

In [ ]:
# Single-patch Mahalanobis comparison
samples = np.random.randn(1000, 3) @ np.array([[2.0, 0.3, 0.1], [0.3, 1.0, 0.0], [0.1, 0.0, 0.5]])
samples += np.array([1.0, -0.5, 2.0])

mu_ours, sinv_ours = fit_gaussian(samples, ridge=0.0)

cov_sk = EmpiricalCovariance().fit(samples)
sinv_sk = np.linalg.inv(cov_sk.covariance_)

query = np.array([3.0, -1.0, 1.0])
d_ours = mahalanobis_distance(query, mu_ours, sinv_ours)
d_scipy = scipy_mahalanobis(query, cov_sk.location_, sinv_sk)
d_sklearn = np.sqrt(cov_sk.mahalanobis(query[None, :])[0])

print(f'Ours       : {d_ours:.6f}')
print(f'scipy      : {d_scipy:.6f}')
print(f'sklearn    : {d_sklearn:.6f}')
print('\nAll three should agree to ~1e-4.')

---
## Part 8: Reproducing the Voting Decision / Voting 결정 재현

논문 Fig. 1의 **voting 블록** — cosine 기반 분류와 Mahalanobis 기반 분류를 결합 — 을 구현한다. / Reproduce the voting block from Fig. 1 that combines cosine and Mahalanobis verdicts.

In [ ]:
def voting_classify(
    z_test: np.ndarray,
    z_ref_cme: np.ndarray,
    z_ref_nc: np.ndarray,
) -> np.ndarray:
    """Combine reference-based cosine and Mahalanobis classifiers via voting.

    Args:
        z_test: Test embeddings, shape (M, d).
        z_ref_cme: CME reference embeddings, shape (R_c, d).
        z_ref_nc: Non-CME reference embeddings, shape (R_nc, d).

    Returns:
        Labels of shape (M,): 1 for CME, 0 for non-CME.
    """
    # Branch 1: mean cosine distance
    y_cos = classify_by_reference(z_test, z_ref_cme, z_ref_nc)

    # Branch 2: Mahalanobis to each reference distribution
    mu_c, sinv_c = fit_gaussian(z_ref_cme, ridge=1e-3)
    mu_nc, sinv_nc = fit_gaussian(z_ref_nc, ridge=1e-3)
    d_c = mahalanobis_distance(z_test, mu_c, sinv_c)
    d_nc = mahalanobis_distance(z_test, mu_nc, sinv_nc)
    y_maha = (d_c <= d_nc).astype(int)

    # Voting (2 branches): tie → pick cosine (deterministic fallback)
    votes = y_cos + y_maha
    return np.where(votes >= 2, 1, np.where(votes == 0, 0, y_cos))


y_vote = voting_classify(z_test, ref_cme, ref_nc)

from sklearn.metrics import classification_report

print('=== Cosine-only ===')
print(classification_report(y_test, classify_by_reference(z_test, ref_cme, ref_nc),
                            target_names=['non-CME', 'CME']))
print('=== Voting (Cosine + Mahalanobis) ===')
print(classification_report(y_test, y_vote, target_names=['non-CME', 'CME']))

---
## Summary / 요약

이 노트북에서 CMEGNets의 세 가지 수학적 빌딩 블록을 NumPy로 구현하고 합성 데이터에서 검증했다. / We implemented and validated the three mathematical building blocks of CMEGNets on synthetic data.

| 구성요소 / Component | 구현 / Implementation | 검증 / Validation |
|---|---|---|
| NT-Xent contrastive loss | NumPy — `nt_xent_loss` | Aligned positive pairs ≪ random loss; sweeps with τ |
| Reference-based cosine classifier | NumPy — `classify_by_reference` | Synthetic 2-cluster data, accuracy ≈ 1.0 |
| Per-pixel Mahalanobis heat map | NumPy — `pixel_mahalanobis_map` | Synthetic quiet-Sun + CME; Dice > 0.5 at best threshold |
| Mini U-Net architecture | PyTorch — `MiniUNet` | Forward-pass output shape matches input; param count reported |
| Voting classifier | NumPy — `voting_classify` | Matches/exceeds cosine-only on separable data |

### 한계 / Caveats

- **SimCLR full pre-training**은 2.3M LASCO 프레임에 대해 GPU cluster 규모가 필요 — 여기서는 손실 함수만 검증. / SimCLR pre-training on 2.3M LASCO frames needs a GPU cluster; we only verified the loss.
- **합성 "LASCO"** 이미지는 해석 목적 장난감이다 — 실제 photometric fidelity는 보장되지 않는다. / Toy synthetic LASCO — not photometrically faithful.
- **ResNet-18 feature vs handcrafted feature**: 논문은 d=3 학습된 feature를 사용; 우리는 intensity/local-mean/local-std 3채널로 대체. 공간적 prior가 빠져 있음. / Paper uses learned d=3 features; we use an intensity/mean/std handcraft substitute.
- **U-Net 훈련**은 본 노트북 범위 밖 — pseudo-label 생성 → training loop → Dice curve는 future work. / U-Net training is beyond this notebook's scope.

### 다음 단계 / Next Steps

| Next Paper / 다음 논문 | Relationship / 관계 |
|---|---|
| Chen et al. 2020 — SimCLR | CMEGNets의 contrastive 토대 — full pretraining 재현 / Full contrastive pre-training. |
| Ronneberger et al. 2015 — U-Net | Segmentation head의 기원 — 의사 라벨 훈련 실습 / Training the head on pseudo-labels. |
| He et al. 2016 — ResNet | Backbone 대체 및 비교 / Backbone replacement/comparison. |
| Lin et al. 2025 (SW #40) — 3D CME reconstruction | CMEGNets 출력(2D mask)을 dual-view 3D 재구성 입력으로 연결 / Link 2D masks to dual-view 3D reconstruction. |
| Pricopi et al. 2022 — Geoeffectiveness ML | Segmented mask → kinematic → geoeffectiveness forecasting / Downstream geoeffectiveness ML. |